In [ ]:
"""
For Google Colab.
"""

# Clone repo
%cd /content
!git clone https://github.com/AHHHHHH0-0/Extending-CoFinDiff.git
%cd Extending-CoFinDiff

# Open terminal and switch branch

## Import

In [ ]:
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from denoiser.unet_model import UNetDenoiser
from preprocessing.condition_encoder import MicroConditionEncoder
from preprocessing.haar_wavelet import HaarWaveletTransform
from diffusion.diffusion import Diffusion
from config import project_config, diffusion_config, preprocess_config

## Config

In [ ]:
# Seed and device
SEED = project_config.SEED
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    
DEVICE = project_config.DEVICE
CHECKPOINT_PATH = 'models/checkpoints/best_model.pt'

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

## Load Models

In [ ]:
# Models
denoiser = UNetDenoiser(in_channels=1).to(DEVICE)
micro_encoder = MicroConditionEncoder().to(DEVICE)
diffusion = Diffusion()
haar = HaarWaveletTransform()

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
denoiser.load_state_dict(checkpoint['model_state_dict'])
micro_encoder.load_state_dict(checkpoint['micro_encoder_state_dict'])

denoiser.eval()
micro_encoder.eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.6f}")
print(f"Denoiser parameters: {denoiser.get_num_parameters():,}")

## Set Conditioning

In [ ]:
# Generationg configs
N_SAMPLES = 16

TREND = 0.0          
REALIZED_VOL = 50.0 
INTEREST_RATE = 0.05    
VOLATILITY_INDEX = 20.0 

GUIDANCE_SCALE = diffusion_config.GUIDANCE_SCALE 

# Build batch tensors
trend_t          = torch.full((N_SAMPLES, 1), TREND,            dtype=torch.float32, device=DEVICE)
realized_vol_t   = torch.full((N_SAMPLES, 1), REALIZED_VOL,     dtype=torch.float32, device=DEVICE)
interest_rate_t  = torch.full((N_SAMPLES, 1), INTEREST_RATE,    dtype=torch.float32, device=DEVICE)
volatility_idx_t = torch.full((N_SAMPLES, 1), VOLATILITY_INDEX, dtype=torch.float32, device=DEVICE)

# Encode micro conditions
with torch.no_grad():
    micro_cond_tokens = micro_encoder(trend_t, realized_vol_t)

# Stack macro scalars
macro_emb = torch.cat([interest_rate_t, volatility_idx_t], dim=1)

print(f"micro_cond_tokens shape: {micro_cond_tokens.shape}")
print(f"macro_emb shape:         {macro_emb.shape}")

## Generate

In [ ]:
H, W = preprocess_config.TARGET_SHAPE  # (8, 8)
shape = (N_SAMPLES, 1, H, W)

print(f"Sampling {N_SAMPLES} samples with shape {shape} ...")
print(f"Diffusion timesteps: {diffusion_config.TIMESTEPS}")
print(f"Guidance scale: {GUIDANCE_SCALE}")

samples_2d = diffusion.sample(
    model=denoiser,
    shape=shape,
    micro_cond_tokens=micro_cond_tokens,
    macro_emb=macro_emb,
    guidance_scale=GUIDANCE_SCALE,
)
# samples_2d: (N_SAMPLES, 1, H, W)

print(f"\nGenerated samples shape: {samples_2d.shape}")
print(f"Value range: [{samples_2d.min().item():.4f}, {samples_2d.max().item():.4f}]")

## Post-process: Inverse Haar → 1D Log Returns

In [ ]:
# Remove channel dim: (N_SAMPLES, 1, H, W) -> (N_SAMPLES, H, W)
samples_squeezed = samples_2d.squeeze(1).cpu()

# Inverse Haar wavelet: (N_SAMPLES, H, W) -> (N_SAMPLES, T)
samples_1d = haar.inverse(samples_squeezed)

print(f"1D log-return sequences shape: {samples_1d.shape}")
print(f"\nSummary statistics across all samples:")
print(f"  mean:  {samples_1d.mean().item():.6f}")
print(f"  std:   {samples_1d.std().item():.6f}")
print(f"  min:   {samples_1d.min().item():.6f}")
print(f"  max:   {samples_1d.max().item():.6f}")

## Visualize

In [ ]:
N_PLOT = min(8, N_SAMPLES)

fig, axes = plt.subplots(2, N_PLOT // 2, figsize=(16, 5), sharex=True)
axes = axes.flatten()

for i in range(N_PLOT):
    returns = samples_1d[i].numpy()
    axes[i].plot(returns, linewidth=0.8)
    axes[i].axhline(0, color='gray', linewidth=0.5, linestyle='--')
    axes[i].set_title(f"Sample {i + 1}", fontsize=9)
    axes[i].set_ylabel("Log return", fontsize=7)
    axes[i].set_xlabel("Timestep", fontsize=7)

plt.suptitle(
    f"Generated Log Returns  |  trend={TREND}  realized_vol={REALIZED_VOL}  "
    f"rate={INTEREST_RATE}  vix={VOLATILITY_INDEX}",
    fontsize=10
)
plt.tight_layout()
plt.show()

## Save Generated Samples

In [ ]:
output_dir = 'data/generated'
output_dir.mkdir(parents=True, exist_ok=True)

generated = {
    'conditions': {
        'trend': TREND,
        'realized_vol': REALIZED_VOL,
        'interest_rate': INTEREST_RATE,
        'volatility_index': VOLATILITY_INDEX,
        'guidance_scale': GUIDANCE_SCALE,
    },
    'samples': samples_1d.numpy().tolist(),  # list of N_SAMPLES sequences of length T
}

output_path = output_dir / 'generated_returns.json'
with open(output_path, 'w') as f:
    json.dump(generated, f)

print(f"Saved {N_SAMPLES} samples to {output_path}")